### Agent Implementations: self-play

#### verS1 stands for S: small - size of board (4x5 board - 16 pieces), 1: self-play
*Note: all configs, parameters and hyperparameters used in this phase are available in "../implementations/verS1_configs.txt"

versions guide:

- X - {0: no, 1: yes}
- S1XYYYY, X - Is Double Net
- S1YXYYY, X - Is Dueling Net
- S1YYXYY, X - Is Residual Net
- S1YYYXY, X - Is Canonical
- S1YYYYX, X - Will do reward shaping

#### Canonical vs Absolute Perspective

Log analysis:

| ver | label | eval max Q | eval min Q | avg Q (first eval) | avg Q (last eval) | max eval TD error | best eval win% white | best eval win% black | end eval win% white | end eval win% black | end train win gap | best train draw% (lowest) | **end train draw% (last)** |
|---|---|---|---|---|---|---|----------------------|----------------------|---------------------|---------------------|-------------------|---|---|
| S110110 | Canonical Residual DDQN | 0.5644 | -0.8632 | 0.0083 | 0.2478 | 1.0539 | 88.5%                | 83.0%                | 86.0%               | 83.0%               | 7.6%              | 25.6% | **25.6%** |
| S110100 | Absolute Perspective Residual DDQN | 0.3757 | -1.0053 | -0.0080 | 0.1642 | 1.0251 | 86.5%                | 77.5%                | 84.0%               | 77.5%               | 6.3%              | 30.3% | **30.3%** |

Regardless of state representation of the board, to always be flipped as white perspective or represented "as it is" from absolute perspective and with additional signal - who is on turn, my assumption is that regardless of choice, color bias is existing because size of the board. In other words when average number of env steps per game is 54 -> 30 that start one move of white matters more, but that is only hypothesis. But even of this, canonical version managed to lower decisive draws by more than a half from baseline and showed slightly better results on evaluations against random opponent than absolute perspective implementation did. Every further implementation will include canonical version as base.

#### Further Training (self-play vs no-self-play)

As shown in *"verS0.ipnyb"* Residual DDQN that is trained against random have 86% win rate as white on evaluation against random which is same as Residual DDQN that is self-play evaluation win rate as white against random. But Residual DDQN trained against random performed worse than certain implementations that had worse evaluation win rate as white against random. So before going deeper into training I would like to check how self-play performs against random opponent training version.

In [1]:
from agent.grenight_agent import GrenightAgent
from agent.helpers.load_checkpoint import load_checkpoint
from agent.helpers.agents_duel import test_agents

self_play = GrenightAgent(True, True, False, True, 12, 5, 4, 420)
load_checkpoint(self_play, "S110110", 10_000)

no_self_play = GrenightAgent(False, True, False, True, 12, 5, 4, 440)
load_checkpoint(no_self_play, "S010100", 10_000)

print(test_agents(self_play, no_self_play, "self-play", "no_self_play", True, False))
print()
print(test_agents(no_self_play, self_play, "no_self_play", "self-play", False, True))

white=self-play, black=no_self_play
Outcomes: Counter({'white_win': 840, 'draw': 149, 'black_win': 11})

white=no_self_play, black=self-play
Outcomes: Counter({'black_win': 902, 'draw': 84, 'white_win': 14})


Unlike no-self-play which was only good for random opponent and bad against trained one, self-play was producing good results in both scenarios. In other words, implementation that is trained against random opponent had extreme loss on both seats, with that given every further training will include self-play as base, and I am ranking self-play to be first criteria for results-chasing choice.